# PathSnap: KNN GPS-to-road matching baseline

This Colab notebook trains a simple K-nearest-neighbors (KNN) **candidate ranker**. For each GPS point, it considers nearby road arcs and scores which candidate is most plausible.

**Label limitation:** the raw target is an ordered list of route arc IDs, not a GPS-row label. For this baseline, a GPS point's proxy target is the geometrically nearest arc in its reference route. Validation metrics below measure agreement with that proxy; they are not exact route-level ground truth. This baseline does not yet enforce a continuous route through the road graph.

**GPU note:** scikit-learn's `KNeighborsClassifier` runs on CPU and does not benefit from Colab's GPU runtime. A GPU runtime is fine to select, but this implementation will use CPU. KNN stores examples rather than learning GPU-accelerated weights.

## 1. Import Required Libraries

Install small missing dependencies, import the project functions, and check whether Colab exposes a GPU. The KNN estimator itself will still run on CPU.

In [ ]:
%pip install -q scikit-learn scipy joblib

from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    print("GPU visible to Colab:", GPU_AVAILABLE)
    if GPU_AVAILABLE:
        print(torch.cuda.get_device_name(0))
except ImportError:
    GPU_AVAILABLE = False
    print("PyTorch not installed; GPU status not checked.")

print("Note: scikit-learn KNN uses CPU regardless of GPU availability.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy/clone this repository into your Drive first, or update these paths to
# wherever you put the repository and its data/Student folder directory.
PROJECT_DIR = Path('/content/drive/MyDrive/machine-learning-hackathon')
DATA_ROOT = PROJECT_DIR / 'data' / 'Student folder'
OUTPUT_DIR = PROJECT_DIR / 'results' / 'knn'

if not (PROJECT_DIR / 'src' / 'knn_model.py').exists():
    raise FileNotFoundError(
        f"Project source not found at {PROJECT_DIR}. Set PROJECT_DIR to the "
        "folder containing src/ and data/."
    )
sys.path.insert(0, str(PROJECT_DIR))

from src.data_processing import audit_split, iter_trip_directories, load_trip
from src.knn_model import (
    FEATURE_COLUMNS,
    build_knn_model,
    make_candidate_examples,
    make_dataset,
    rank_candidates,
    score_proxy_labels,
)

print('Project:', PROJECT_DIR)
print('Dataset:', DATA_ROOT)
print('Output:', OUTPUT_DIR)

## 2. Load and Explore GPS Data

The data is organized by trip. Track rows are GPS samples, node rows define node IDs, arc rows define directed connections between node IDs, and route values are arc IDs. These row counts are not expected to match.

In [ ]:
train_audit = audit_split(DATA_ROOT / 'train')
test_audit = audit_split(DATA_ROOT / 'student_test')
print('Train trips:', len(train_audit), '| Test trips:', len(test_audit))
display(train_audit.head())
print('\nTrack row-count summary:')
display(train_audit[['track_rows', 'node_rows', 'arc_rows', 'route_rows', 'one_second_interval_fraction']].describe())
print('\nTrack file suffixes:', sorted(train_audit.track_file.unique()))

example_trip_dir = next(iter_trip_directories(DATA_ROOT / 'train'))
example_trip = load_trip(example_trip_dir, include_route=True)
print('Example trip:', example_trip.trip_id)
print('Track:', example_trip.track.shape, '| nodes:', example_trip.nodes.shape,
      '| arcs:', example_trip.arcs.shape, '| route arcs:', len(example_trip.route))
display(example_trip.track.head())
display(example_trip.arcs.head())
print('First route arc IDs:', example_trip.route.head(10).tolist())

In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(example_trip.track.longitude, example_trip.track.latitude, linewidth=1, label='GPS observations')
plt.scatter(example_trip.track.longitude.iloc[::30], example_trip.track.latitude.iloc[::30], s=12)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title(f'Raw GPS trajectory — trip {example_trip.trip_id}')
plt.axis('equal')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## 3. Preprocess GPS Data

The shared loader removes non-finite/out-of-range coordinates, sorts by timestamp, and drops duplicate timestamps while preserving source-row IDs. It validates arc endpoints and route IDs. We do not apply an arbitrary speed cutoff: noisy jumps are part of the task and should be handled by the matcher rather than silently discarded.

In [ ]:
print('Cleaned GPS rows:', len(example_trip.track))
print('Timestamp duplicates after cleaning:', int(example_trip.track.timestamp_s.duplicated().sum()))
print('Coordinate ranges:')
display(example_trip.track[['longitude', 'latitude', 'timestamp_s']].describe())

delta_seconds = example_trip.track.timestamp_s.diff().dropna()
print(f'Exactly 1-second sampling intervals: {(delta_seconds == 1).mean():.1%}')

## 4. Feature Engineering for KNN

The model ranks nearby road arcs for each GPS point. Pair features are point-to-segment distance, heading difference, arc length, projection along the arc, and GPS motion (speed, step distance, heading validity). Features are standardized inside the model pipeline. The selected candidate is the nearest route arc geometrically; alternative candidates are nearby arcs.

In [ ]:
example_candidates = make_candidate_examples(
    example_trip,
    training=True,
    max_observations=100,
    max_candidates=20,
    max_negatives=8,
)
print('Candidate feature columns:', FEATURE_COLUMNS)
print('Candidate rows:', len(example_candidates))
display(example_candidates.head())
print('Positive proxy examples:', int(example_candidates.label.sum()))

## 5. Split Data into Train and Test Sets

Split **by trip ID**, not by individual GPS rows, to avoid putting samples from the same journey in both subsets. The held-out trips below form validation data for model selection. The separate `student_test` set is not used for training or validation.

In [ ]:
all_train_dirs = list(iter_trip_directories(DATA_ROOT / 'train'))
all_train_ids = [path.name for path in all_train_dirs]
fit_ids, validation_ids = train_test_split(
    all_train_ids, test_size=0.20, random_state=42
)
train_dirs_by_id = {path.name: path for path in all_train_dirs}

print(f'Fit trips: {len(fit_ids)} | Validation trips: {len(validation_ids)}')

# Runtime controls: up to 500 evenly spaced GPS points per trip, 20 nearby arc
# candidates, and at most 8 negative candidates per GPS point.
MAX_OBSERVATIONS_PER_TRIP = 500
MAX_CANDIDATES = 20
MAX_NEGATIVES = 8

fit_trips = [load_trip(train_dirs_by_id[trip_id], include_route=True) for trip_id in fit_ids]
validation_trips = [load_trip(train_dirs_by_id[trip_id], include_route=True) for trip_id in validation_ids]

train_examples = make_dataset(
    fit_trips, training=True,
    max_observations=MAX_OBSERVATIONS_PER_TRIP,
    max_candidates=MAX_CANDIDATES,
    max_negatives=MAX_NEGATIVES,
)
validation_examples = make_dataset(
    validation_trips, training=True,
    max_observations=MAX_OBSERVATIONS_PER_TRIP,
    max_candidates=MAX_CANDIDATES,
    max_negatives=MAX_NEGATIVES,
)

print('Training candidate rows:', len(train_examples),
      '| positive:', int(train_examples.label.sum()))
print('Validation candidate rows:', len(validation_examples),
      '| positive:', int(validation_examples.label.sum()))

## 6. Train KNN Model

The model is a distance-weighted KNN classifier with standardized input features. Each training observation contributes one positive route-arc candidate and nearby negative candidates. Change `N_NEIGHBORS` to experiment with the neighborhood size; use trip-level validation below to compare settings.

In [ ]:
N_NEIGHBORS = 11
knn_model = build_knn_model(n_neighbors=N_NEIGHBORS)
knn_model.fit(
    train_examples[FEATURE_COLUMNS].to_numpy(dtype=np.float32),
    train_examples.label.to_numpy(dtype=np.int8),
)
print('KNN fit complete.')

## 7. Evaluate Model Performance

Report candidate-level precision/recall/F1 and confusion matrix, plus top-1 arc agreement with the proxy target per GPS point. These are **proxy-label validation metrics**. They should not be presented as exact map-matching accuracy without observation-level ground truth.

In [ ]:
validation_x = validation_examples[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
validation_y = validation_examples.label.to_numpy(dtype=np.int8)
validation_pred = knn_model.predict(validation_x)
proxy_metrics = score_proxy_labels(knn_model, validation_examples)

print('Proxy validation metrics:')
for name, value in proxy_metrics.items():
    print(f'  {name}: {value}')
print('\nCandidate-level report:')
print(classification_report(validation_y, validation_pred, labels=[0, 1], zero_division=0))

cm = confusion_matrix(validation_y, validation_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=['other candidate', 'proxy target']).plot(cmap='Blues', values_format='d')
plt.title('Validation candidate classification (proxy labels)')
plt.show()

In [ ]:
# Visual comparison for one held-out trip: GPS track, proxy target arcs, and KNN-ranked arcs.
ranked_validation = rank_candidates(knn_model, validation_examples)
plot_trip_id = validation_ids[0]
plot_trip = next(trip for trip in validation_trips if trip.trip_id == plot_trip_id)
plot_predictions = ranked_validation[ranked_validation.trip_id == plot_trip_id].head(60)
plot_targets = validation_examples[
    (validation_examples.trip_id == plot_trip_id)
][['observation_id', 'target_arc_id']].drop_duplicates('observation_id')
plot_rows = plot_predictions.merge(plot_targets, on='observation_id', how='inner')

plt.figure(figsize=(9, 8))
plt.plot(plot_trip.track.longitude, plot_trip.track.latitude, color='0.7', linewidth=1, label='GPS track')
for _, row in plot_rows.iterrows():
    pred_arc = plot_trip.arcs.loc[int(row.arc_id)]
    target_arc = plot_trip.arcs.loc[int(row.target_arc_id)]
    pred_nodes = plot_trip.nodes.loc[[int(pred_arc.from_node_id), int(pred_arc.to_node_id)]]
    target_nodes = plot_trip.nodes.loc[[int(target_arc.from_node_id), int(target_arc.to_node_id)]]
    plt.plot(pred_nodes.longitude, pred_nodes.latitude, color='tab:red', alpha=0.35, linewidth=1)
    plt.plot(target_nodes.longitude, target_nodes.latitude, color='tab:green', alpha=0.3, linewidth=1)
plt.plot([], [], color='tab:red', label='KNN top-ranked arcs')
plt.plot([], [], color='tab:green', label='Proxy target arcs')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title(f'Validation trip {plot_trip_id}: sample of arc candidates')
plt.axis('equal')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 8. Save Trained KNN Model

After validation, refit the selected KNN configuration using all training trips and save the estimator, feature names, and settings. The optional final cell produces one ranked arc prediction per test GPS observation; graph-level route smoothing is still a separate next step.

In [ ]:
all_training_trips = fit_trips + validation_trips
final_examples = make_dataset(
    all_training_trips, training=True,
    max_observations=MAX_OBSERVATIONS_PER_TRIP,
    max_candidates=MAX_CANDIDATES,
    max_negatives=MAX_NEGATIVES,
)
final_knn_model = build_knn_model(n_neighbors=N_NEIGHBORS)
final_knn_model.fit(
    final_examples[FEATURE_COLUMNS].to_numpy(dtype=np.float32),
    final_examples.label.to_numpy(dtype=np.int8),
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_path = OUTPUT_DIR / 'pathsnap_knn.joblib'
joblib.dump(
    {
        'model': final_knn_model,
        'feature_columns': FEATURE_COLUMNS,
        'n_neighbors': N_NEIGHBORS,
        'max_candidates': MAX_CANDIDATES,
        'label_definition': 'nearest reference-route arc by point-to-segment distance',
        'validation_proxy_metrics': proxy_metrics,
    },
    model_path,
)
with open(OUTPUT_DIR / 'validation_proxy_metrics.json', 'w') as f:
    json.dump(proxy_metrics, f, indent=2)
print('Saved model:', model_path)
print('Saved validation metrics:', OUTPUT_DIR / 'validation_proxy_metrics.json')

In [ ]:
# Optional inference on student_test: no route labels are loaded or used.
test_prediction_frames = []
for test_dir in iter_trip_directories(DATA_ROOT / 'student_test'):
    test_trip = load_trip(test_dir, include_route=False)
    test_candidates = make_candidate_examples(
        test_trip, training=False,
        max_observations=None,  # predict for every cleaned GPS observation
        max_candidates=MAX_CANDIDATES,
    )
    trip_predictions = rank_candidates(final_knn_model, test_candidates)
    test_prediction_frames.append(trip_predictions)

test_predictions = pd.concat(test_prediction_frames, ignore_index=True)
prediction_path = OUTPUT_DIR / 'student_test_arc_predictions.csv'
test_predictions.to_csv(prediction_path, index=False)
print('Saved predictions:', prediction_path)
display(test_predictions.head())